In [1]:
import numpy as np
import os, glob
import matplotlib.pyplot as plt
from netCDF4 import Dataset, MFDataset
from datetime import datetime, timedelta
import pandas as pd
import xarray as xr
import cftime
from sklearn.linear_model import LinearRegression
import warnings
from sklearn.metrics import r2_score

In [2]:
warnings.simplefilter("ignore")

In [3]:
def get_satellite_predictors(X,var_sat):
    sat_vars = {'tbright':'brightness_temperature_clear',
                'pct_below':'percent_pixel_below_threshold'}
    #
    loadvar = np.empty([len(var_sat),len(X.variables['latitude'][:].data),
                           len(X.variables['longitude'][:].data)])
    #
    for i in np.arange(0,len(var_sat)):
        # print(var_sel)
        var_sel = var_sat[i]
        varname_sel = sat_vars[var_sel]
        ivar = X[varname_sel].squeeze()
        loadvar[i,:,:] = ivar
    return loadvar

In [4]:
def get_sst_predictors(X,var_sst):
    sst_vars = {'sst':'sea_surface_temperature'}
    loadvar = np.empty([len(sst_vars),len(X.variables['latitude'][:].data),
                        len(X.variables['longitude'][:].data)])
    for i in np.arange(0,len(var_sst)):
        var_sel = var_sst[i]
        varname_sel = sst_vars[var_sel]
        ivar = X[varname_sel].squeeze()
        loadvar[i,:,:] = ivar
    return loadvar

In [5]:
model_vars = {'RH':'relative_humidity',
              'vort':'vorticity',
              'div':'divergence',
              'temp':'temperature',
              'u':'u-wind',
              'v':'v-wind',
              'z':'geopotential_height'}

In [6]:
def get_model_predictors(X,var_model):
    model_vars = {'RHmd':'relative_humidity',
              'vortl':'vorticity',
              'divl':'divergence',
              'divu':'divergence',
              'temp':'temperature'}
    model_var_levels = {'RHmd':[500,600,700],
                    'vortl':[850],
                    'divl':[850],
                    'divu':[200],
                    'temp':[200]}
    model_var_rad = {'RHmd':[0,1000],
                 'vortl':[0,1000],
                 'divl':[0,1000],
                 'divu':[0,1000],
                 'temp':[0,1000]}
    # 
    loadvar = np.empty([len(var_model),
                        len(X.variables['level'][:].data),
                        len(X.variables['latitude'][:].data),
                        len(X.variables['longitude'][:].data),
                        len(X.variables['region'][:].data)])
    #
    for i in np.arange(0,len(var_model)):
        # print(var_sel)
        var_sel = var_model[i]
        varname_sel = model_vars[var_sel]
        ivar = X[varname_sel].squeeze()
        
        # lev_sel = model_var_levels[var_sel]
        # reg_sel = model_var_rad[var_sel]
        #print(varname_sel,lev_sel)
        # 
        # levels = boop.variables['level']
        # if lev_sel:
          #  if reg_sel:
           #     #ivar = X[varname_sel].sel(level=lev_sel,region=reg_sel).mean(dim=["level","region"]).squeeze()
           #     ivar = X[varname_sel].
            # else:
                #ivar = X[varname_sel].sel(level=lev_sel).mean(dim="level").squeeze()
       #  else:
         #   if reg_sel:
          #      ivar = X[varname_sel].sel(region=reg_sel).mean(dim="region").squeeze()
           # else:
             #   ivar = X[varname_sel].squeeze()
            
        loadvar[i,:,:,:,:] = ivar
    return loadvar

In [7]:
def get_model_predictors_full(X,var_model):
    model_vars = {'RH':'relative_humidity',
              'vort':'vorticity',
              'div':'divergence',
              'u':'u_wind',
              'v':'v_wind',
              'temp':'temperature'}
    loadvar = np.empty([len(var_model),
                        len(X.variables['level'][:].data),
                        len(X.variables['region'][:].data)])
    for i in np.arange(0,len(var_model)):
        # print(var_sel)
        var_sel = var_model[i]
        varname_sel = model_vars[var_sel]
        ivar = X[varname_sel].squeeze().mean(axis=(1,2))
        loadvar[i,:,:] = ivar
    return loadvar

In [8]:
datacat_dict = {'satellite':'sat',
                'storm':'storm',
                'derived':'deriv',
                'model':'model',
                'sst':'sst'}
#
datacat_vars = {'satellite':['tbright','pct_below'],
                'derived':['shrg','tanom','hmax','wvel'],
                'model':['RH','vort','div','u','v','temp'],
                'sst':['sst']}
#
data_cat = ['model']

In [9]:
start_date = '2016-01-01'
end_date = '2017-01-01'

In [10]:
days_back = 0
forecast_date = pd.date_range(start=start_date,end=end_date,freq='6H')

In [11]:
for iforecast in forecast_date:
    data_dates = iforecast - pd.Timedelta(days_back,'D')
    if np.mod(iforecast.day,7) == 0:
        print('forecasting for ',iforecast,'; data from ',data_dates)
    #
    yr_sel = data_dates.year
    mon_sel = data_dates.month
    day_sel = data_dates.day
    hr_sel = data_dates.hour
    #
    mon_sel_str = str(mon_sel) if mon_sel > 9 else '0'+str(mon_sel)
    day_sel_str = str(day_sel) if day_sel > 9 else '0'+str(day_sel)
    hr_sel_str = str(hr_sel) if hr_sel > 9 else '0'+str(hr_sel)
    # 
    variables_list = []
    full_predictors = []
    for idata in data_cat:
        #print(idata)
        data_dir = '/mnt/tcnas08/cslocum/TCFPv4/devdat/v1p0/{datatype}/1p00/'.format(datatype=idata)
        fpath = data_dir+'{year}/{mon}{day}/'.format(year=yr_sel,mon = mon_sel_str,day = day_sel_str)
        #
        fname = 'TCFP-devdat-pred-{dcat}_v4r0_blend_s{year}{mon}{day}{hr}00000_e{year}{mon}{day}{hr}00000_*.nc'.format(dcat=datacat_dict[idata],
        year=yr_sel,mon=mon_sel_str,day=day_sel_str,hr=hr_sel_str)
        if not os.listdir(fpath):
            continue
        # 
        if not glob.glob(fpath+fname):
            continue
        else:
            xdata = xr.open_mfdataset(fpath+fname)
            var_ALL = datacat_vars[idata]
            # X_model = get_satellite_predictors(xdata,var_ALL)
            # X_model = get_sst_predictors(xdata,var_ALL)
            X_model = get_model_predictors_full(xdata,var_ALL)
           #  lat = xdata.variables['latitude'][:]
           # lon = xdata.variables['longitude'][:]
            #
           #  variables_list = variables_list+var_ALL
            if idata == data_cat[0]:
                full_predictors = X_model
            else:
                full_predictors = np.append(full_predictors,X_model,axis=0)
        # Check if empty 
    if len(full_predictors) == 0:
       print('no data for this forecast')
       full_predictors = np.empty((full_predictors_ALL.shape[0],full_predictors_ALL.shape[1],full_predictors_ALL.shape[2]))*np.nan
    if iforecast == forecast_date[0]:
        full_predictors_ALL = full_predictors
    elif iforecast == forecast_date[1]:
        full_predictors_ALL = np.stack([full_predictors_ALL,full_predictors],axis=3)
    else:
        full_predictors_ALL = np.append(full_predictors_ALL,np.expand_dims(full_predictors,axis=3),axis=3)

forecasting for  2016-01-07 00:00:00 ; data from  2016-01-07 00:00:00
forecasting for  2016-01-07 06:00:00 ; data from  2016-01-07 06:00:00
forecasting for  2016-01-07 12:00:00 ; data from  2016-01-07 12:00:00
forecasting for  2016-01-07 18:00:00 ; data from  2016-01-07 18:00:00
forecasting for  2016-01-14 00:00:00 ; data from  2016-01-14 00:00:00
forecasting for  2016-01-14 06:00:00 ; data from  2016-01-14 06:00:00
forecasting for  2016-01-14 12:00:00 ; data from  2016-01-14 12:00:00
forecasting for  2016-01-14 18:00:00 ; data from  2016-01-14 18:00:00
forecasting for  2016-01-21 00:00:00 ; data from  2016-01-21 00:00:00
forecasting for  2016-01-21 06:00:00 ; data from  2016-01-21 06:00:00
forecasting for  2016-01-21 12:00:00 ; data from  2016-01-21 12:00:00
forecasting for  2016-01-21 18:00:00 ; data from  2016-01-21 18:00:00
forecasting for  2016-01-28 00:00:00 ; data from  2016-01-28 00:00:00
forecasting for  2016-01-28 06:00:00 ; data from  2016-01-28 06:00:00
forecasting for  201

(3, 13, 15)

In [12]:
X_model.shape

(5, 13, 91, 360, 15)

In [ ]:
X_test = full_predictors_ALL[1,:,:,:].squeeze()
X2 = np.nanmean(X_test,axis=(0,1))

In [ ]:
n = len(X2)
var = np.nanvar(X2)
xstd = X2 - np.nanmean(X2)
r = np.correlate(xstd,xstd,mode='full')[-n:]

In [ ]:
xdata

In [ ]:
rho = r/(var*(np.arange(n,0,-1)))
doop = np.where(rho < (1/np.e))


In [ ]:
rho[doop[0][0]]

In [ ]:
doop[0][0]/4

In [ ]:
plt.plot(rho)